<a href="https://colab.research.google.com/github/sabarish-3505/self-project/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installation

In [1]:
!pip install -q langgraph langchain langchain-ollama langchain-community \
    langchain-mcp-adapters mcp chromadb langchain-text-splitters pydantic
!sudo apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

def ollama_is_running():
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except Exception:
        return False

if not ollama_is_running():
    print("Starting Ollama server...")
    subprocess.Popen(["ollama", "serve"],
                      stdout=subprocess.DEVNULL,
                      stderr=subprocess.DEVNULL)
    for i in range(30):
        time.sleep(1)
        if ollama_is_running():
            print(f"Server ready after {i+1}s.")
            break
    else:
        raise RuntimeError("Ollama didn't start in 30s -- re-run this cell.")
else:
    print("Ollama already running.")

print("Pulling the Llama 3.1 model... (this may take a minute or two)")

!ollama pull llama3.1
!ollama pull nomic-embed-text

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Server ready after 2s.
Pulling the Llama 3.1 model... (this may take a minute or two)




In [7]:
import os

os.makedirs("sample_docs", exist_ok=True)

docs = {
"bearing_wear.txt": """Failure Mode: Bearing Wear / Bearing Housing Degradation
Applicable Equipment: Pump assembly stations, motor shaft bearings, CB18/CB28 lines

Symptoms:
- Increased vibration amplitude at 1x and 2x shaft rotational frequency
- Audible grinding, rumbling, or high-pitched squealing near the bearing housing
- Localized temperature rise at the bearing housing (>15C above baseline)
- Gradual increase in motor current draw as friction increases

Root Causes:
- Inadequate or contaminated lubrication (most common, ~55% of cases)
- Misalignment between motor shaft and pump shaft inducing radial load
- Bearing fatigue from normal end-of-life wear (typically after 8000-12000 operating hours)
- Ingress of particulate contamination through a damaged seal

Recommended Corrective Actions:
- Replace bearing and inspect housing bore for scoring
- Verify shaft alignment with dial indicator or laser alignment tool within 0.05mm
- Replace lubrication per OEM schedule; check grease contamination
- Required parts: bearing (deep groove ball or angular contact per station spec), grease cartridge
- Required skill: Mechanical Technician - Bearing/Rotating Equipment certified
""",

"seal_leakage.txt": """Failure Mode: Mechanical Seal Leakage
Applicable Equipment: Pump assembly stations, fluid transfer lines

Symptoms:
- Visible fluid weeping or dripping at the pump seal housing
- Gradual pressure drop in the discharge line
- Reduced flow rate reported downstream
- Occasionally accompanied by a faint hissing sound if seal has cracked

Root Causes:
- Seal face wear from abrasive particulate in process fluid
- Improper seal installation torque during assembly
- Thermal shock causing seal face micro-cracking
- Dry-running condition during startup causing seal face burn

Recommended Corrective Actions:
- Replace mechanical seal assembly (primary and secondary seal faces)
- Inspect for particulate contamination upstream, add/replace filtration if needed
- Verify seal installation torque against OEM spec at reassembly
- Required parts: mechanical seal kit (station-specific part number)
- Required skill: Mechanical Technician - Seal/Fluid Systems certified
""",

"misalignment.txt": """Failure Mode: Shaft Misalignment
Applicable Equipment: Motor-pump coupled stations

Symptoms:
- Elevated vibration at 1x and 2x running speed, strongest in the axial direction
- Coupling wear or visible heat discoloration at the coupling
- Uneven bearing wear pattern (one side of bearing wears faster)
- Noise correlated with coupling rotation, not bearing housing specifically

Root Causes:
- Improper installation/assembly alignment during station build or rebuild
- Thermal growth differential between motor and pump not accounted for
- Loose or degraded mounting bolts allowing baseplate shift over time
- Foundation settling or baseplate distortion

Recommended Corrective Actions:
- Perform laser shaft alignment, target within 0.05mm parallel / 0.02 deg angular
- Inspect and torque all mounting bolts to spec
- Inspect coupling for wear, replace if elastomer/spider element degraded
- Required parts: coupling insert/spider (if worn), shim stock
- Required skill: Mechanical Technician - Alignment certified
""",

"motor_overheating.txt": """Failure Mode: Motor Overheating
Applicable Equipment: Drive motors on pump assembly stations

Symptoms:
- Motor casing temperature exceeding rated thermal class threshold
- Thermal protection trips / repeated nuisance shutdowns
- Discoloration or burning smell near motor windings
- Elevated current draw disproportionate to load

Root Causes:
- Blocked or fouled cooling fan / ventilation path
- Sustained overload from downstream mechanical restriction (e.g. bearing drag, misalignment)
- Voltage imbalance across phases
- Winding insulation degradation from age or repeated thermal cycling

Recommended Corrective Actions:
- Clean/inspect cooling fan and ventilation path
- Investigate downstream mechanical load (check for bearing wear/misalignment as root contributor)
- Measure phase voltage balance, correct supply issue if found
- Megger test winding insulation resistance; replace motor if degraded below threshold
- Required parts: cooling fan assembly (if faulty), motor (only if winding failure confirmed)
- Required skill: Electrical Technician - Motor Systems certified
""",
}

for name, content in docs.items():
    with open(f"sample_docs/{name}", "w") as f:
        f.write(content)

print("Sample docs written:", os.listdir("sample_docs"))

Sample docs written: ['motor_overheating.txt', 'bearing_wear.txt', 'misalignment.txt', 'seal_leakage.txt']


In [2]:
%%writefile mcp_server.py
"""MCP server exposing three 'live system' tools that only the Planning
Agent is allowed to call: spare parts inventory, technician scheduling,
and work-order history."""

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("maintenance-ops")

_SPARE_PARTS_DB = {
    "bearing": {"part_no": "BRG-6205-2RS", "on_hand": 6, "lead_time_days": 0},
    "grease cartridge": {"part_no": "LUB-GRS-500", "on_hand": 20, "lead_time_days": 0},
    "mechanical seal": {"part_no": "SEAL-MC-118", "on_hand": 1, "lead_time_days": 3},
    "coupling insert": {"part_no": "CPL-INS-22", "on_hand": 4, "lead_time_days": 0},
    "cooling fan assembly": {"part_no": "MTR-FAN-40", "on_hand": 0, "lead_time_days": 5},
    "motor": {"part_no": "MTR-3PH-2.2KW", "on_hand": 1, "lead_time_days": 7},
}

_TECHNICIANS = [
    {"name": "R. Iyer", "skill": "Mechanical Technician - Bearing/Rotating Equipment", "available_in_hours": 1},
    {"name": "K. Nair", "skill": "Mechanical Technician - Seal/Fluid Systems", "available_in_hours": 4},
    {"name": "P. Deshmukh", "skill": "Mechanical Technician - Alignment", "available_in_hours": 2},
    {"name": "S. Fernandes", "skill": "Electrical Technician - Motor Systems", "available_in_hours": 6},
]

_WORK_ORDER_HISTORY = {
    "CB18-ST3": [
        "WO-2091: Bearing replaced, Station 3, 2025-11-02",
        "WO-1876: Lubrication top-up, Station 3, 2025-08-14",
    ],
    "CB28-ST1": [
        "WO-2210: Seal replacement, Station 1, 2026-01-20",
    ],
}


@mcp.tool()
def lookup_spare_parts(component: str) -> dict:
    """Look up spare-parts inventory for a named component (e.g. 'bearing',
    'mechanical seal', 'coupling insert', 'cooling fan assembly', 'motor').
    """
    key = component.strip().lower()
    if key in _SPARE_PARTS_DB:
        return {"component": key, **_SPARE_PARTS_DB[key]}
    return {"component": key, "part_no": None, "on_hand": 0, "lead_time_days": None,
            "note": "component not found in inventory master"}


@mcp.tool()
def get_technician_availability(skill: str) -> dict:
    """Find the soonest-available technician certified for a given skill."""
    matches = [t for t in _TECHNICIANS if t["skill"].lower() == skill.strip().lower()]
    if not matches:
        return {"skill": skill, "technician": None, "note": "no certified technician found"}
    best = min(matches, key=lambda t: t["available_in_hours"])
    return {"skill": skill, "technician": best["name"], "available_in_hours": best["available_in_hours"]}


@mcp.tool()
def get_work_order_history(equipment_id: str) -> list:
    """Return recent work-order history for a given equipment/station ID
    (e.g. 'CB18-ST3', 'CB28-ST1')."""
    return _WORK_ORDER_HISTORY.get(equipment_id.strip().upper(), [])


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting mcp_server.py


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBED_MODEL = "nomic-embed-text"

def build_vectorstore():
    embeddings = OllamaEmbeddings(model=EMBED_MODEL)
    loader = DirectoryLoader("sample_docs", glob="*.txt", loader_cls=TextLoader)
    raw_docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    chunks = splitter.split_documents(raw_docs)
    return Chroma.from_documents(chunks, embedding=embeddings, persist_directory="./chroma_store")

vectorstore = build_vectorstore()

def get_retriever(k: int = 4):
    return vectorstore.as_retriever(search_kwargs={"k": k})

print("Vector store ready:", vectorstore._collection.count(), "chunks")

/tmp/ipykernel_3619/1824118845.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Vector store ready: 10 chunks


In [4]:
from typing import Annotated, List, Literal, Optional, TypedDict
import operator

Severity = Literal["low", "medium", "high"]

class RootCause(TypedDict):
    cause: str
    confidence: float
    evidence: str

class AgentState(TypedDict, total=False):
    equipment_id: str
    alert_text: str
    severity: Severity
    fault_category: str
    root_causes: List[RootCause]
    diagnostic_confidence: float
    diagnostic_attempts: int
    action_plan: str
    final_report: str
    trace: Annotated[List[str], operator.add]

In [5]:
import json
from typing import List, Union
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from pydantic import BaseModel, Field
import sys
import functools

# --- Patch stdio_client's errlog so it doesn't default to ipykernel's fake
# sys.stderr (which has no real file descriptor and breaks subprocess.Popen
# in Colab). stdio_client is @asynccontextmanager-wrapped, so it has no
# plain __defaults__ to edit — instead we bind errlog via functools.partial
# and patch BOTH the source module and langchain_mcp_adapters.sessions,
# since that module imported the name directly into its own namespace.
import mcp.client.stdio as _mcp_stdio_module
import langchain_mcp_adapters.sessions as _lc_mcp_sessions

if not getattr(_mcp_stdio_module, "_stdio_patched_for_colab", False):
    _real_log_file = open("/tmp/mcp_server_stderr.log", "w")
    _original_stdio_client = _mcp_stdio_module.stdio_client
    _patched_stdio_client = functools.partial(_original_stdio_client, errlog=_real_log_file)

    _mcp_stdio_module.stdio_client = _patched_stdio_client
    _lc_mcp_sessions.stdio_client = _patched_stdio_client

    _mcp_stdio_module._stdio_patched_for_colab = True
    print("Patched stdio_client errlog for Colab.")

LLM_MODEL = "llama3.1"

def _llm(temperature: float = 0.0) -> ChatOllama:
    return ChatOllama(model=LLM_MODEL, temperature=temperature)

# --- 1. Triage Agent ---
class TriageResult(BaseModel):
    severity: str = Field(description="one of: low, medium, high")
    fault_category: str = Field(description="e.g. 'vibration', 'leakage', 'thermal', 'electrical'")
    reasoning: str = Field(description="one sentence justification")

def triage_agent(state: AgentState) -> dict:
    llm = _llm().with_structured_output(TriageResult)
    result = llm.invoke([
        SystemMessage(content=(
            "You are a maintenance triage agent. Classify severity and fault category. "
            "severity=low: cosmetic/no functional impact, log only. "
            "severity=medium: degraded performance, plan maintenance soon. "
            "severity=high: imminent failure/safety risk, needs immediate diagnosis."
        )),
        HumanMessage(content=f"Equipment: {state['equipment_id']}\nAlert: {state['alert_text']}"),
    ])
    return {
        "severity": result.severity,
        "fault_category": result.fault_category,
        "trace": [f"[triage] severity={result.severity} category={result.fault_category} ({result.reasoning})"],
    }

# --- 2. Diagnostic Agent (RAG) ---

class DiagnosisResult(BaseModel):
    root_causes: List[dict] = Field(
        description="list of {cause, confidence, evidence}, ranked most likely first"
    )
    overall_confidence: Union[str, float] = Field(
        description="confidence in top root cause, as a decimal 0-1"
    )


def _coerce_confidence(value) -> float:
    """llama3.1 sometimes ignores the 'float 0-1' instruction and writes
    words like 'high'/'medium'/'low' or percentages instead of a number.
    Coerce whatever it gives us into a 0-1 float instead of crashing."""
    if isinstance(value, (int, float)):
        return max(0.0, min(1.0, float(value)))

    if isinstance(value, str):
        s = value.strip().lower().rstrip("%")
        try:
            f = float(s)
            return max(0.0, min(1.0, f / 100 if f > 1 else f))
        except ValueError:
            pass
        word_map = {
            "very high": 0.9, "high": 0.8,
            "medium": 0.5, "moderate": 0.5,
            "low": 0.3, "very low": 0.15,
            "none": 0.0, "unknown": 0.0,
        }
        for word, score in word_map.items():
            if word in s:
                return score

    return 0.3  # safe default if we truly can't parse it


def diagnostic_agent(state: AgentState) -> dict:
    retriever = get_retriever(k=4)
    docs = retriever.invoke(state["alert_text"])
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    llm = _llm().with_structured_output(DiagnosisResult)
    result = llm.invoke([
        SystemMessage(content=(
            "You are a maintenance diagnostic agent. Propose ranked root causes GROUNDED "
            "ONLY in the retrieved excerpts below, citing evidence phrases. If unsupported, "
            "say so with low confidence. "
            "IMPORTANT: every 'confidence' value, including overall_confidence, MUST be a "
            "plain decimal number between 0 and 1 (e.g. 0.75) — never a word like 'high' or 'low'.\n\n"
            "RETRIEVED EXCERPTS:\n" + context
        )),
        HumanMessage(content=f"Equipment: {state['equipment_id']}\nAlert: {state['alert_text']}\n"
                              f"Fault category from triage: {state.get('fault_category')}"),
    ])

    root_causes = [
        {"cause": rc.get("cause", ""),
         "confidence": _coerce_confidence(rc.get("confidence", 0.0)),
         "evidence": rc.get("evidence", "")}
        for rc in result.root_causes
    ]
    overall_confidence = _coerce_confidence(result.overall_confidence)
    attempts = state.get("diagnostic_attempts", 0) + 1

    return {
        "root_causes": root_causes,
        "diagnostic_confidence": overall_confidence,
        "diagnostic_attempts": attempts,
        "trace": [f"[diagnostic] attempt={attempts} top_cause="
                  f"{root_causes[0]['cause'] if root_causes else 'none'} "
                  f"confidence={overall_confidence:.2f}"],
    }

# --- 3. Planning Agent (MCP tools) ---
async def load_maintenance_ops_tools():
    client = MultiServerMCPClient({
        "maintenance-ops": {
            "command": sys.executable,
            "args": ["mcp_server.py"],
            "transport": "stdio",
        }
    })
    return await client.get_tools()

async def planning_agent(state: AgentState) -> dict:
    tools = await load_maintenance_ops_tools()
    agent = create_react_agent(_llm(), tools)
    top_cause = state["root_causes"][0]["cause"] if state.get("root_causes") else "unknown fault"

    prompt = (
        "You are a maintenance planning agent. Using the tools available (spare parts lookup, "
        "technician availability, work order history), build a concrete action plan.\n\n"
        f"Equipment: {state['equipment_id']}\nProbable root cause: {top_cause}\n"
        f"All candidate causes: {json.dumps(state.get('root_causes', []))}\n\n"
        "Steps: (1) check work order history for this equipment, (2) look up spare parts for "
        "the likely fix, (3) find an available technician with the matching skill. Then write "
        "a short concrete plan referencing the actual part numbers, technician, and availability."
    )
    result = await agent.ainvoke({"messages": [HumanMessage(content=prompt)]})
    return {
        "action_plan": result["messages"][-1].content,
        "trace": ["[planning] action plan generated via MCP tools"],
    }

# --- 4. Report Agent ---
def report_agent(state: AgentState) -> dict:
    if state.get("severity") == "low":
        body = (f"Equipment: {state['equipment_id']}\nAlert: {state['alert_text']}\n"
                 f"Severity: low — logged only, no action required.\n"
                 f"Fault category: {state.get('fault_category')}")
        return {"final_report": body, "trace": ["[report] low-severity, log-only report generated"]}

    llm = _llm(temperature=0.2)
    response = llm.invoke([
        SystemMessage(content="You are a maintenance report agent. Write a concise, actionable "
                               "report for a shift supervisor: severity, likely root cause with "
                               "confidence, and the action plan."),
        HumanMessage(content=(
            f"Equipment: {state['equipment_id']}\nAlert: {state['alert_text']}\n"
            f"Severity: {state.get('severity')}\nFault category: {state.get('fault_category')}\n"
            f"Root causes: {json.dumps(state.get('root_causes', []))}\n"
            f"Diagnostic confidence: {state.get('diagnostic_confidence')}\n"
            f"Action plan: {state.get('action_plan')}"
        )),
    ])
    return {"final_report": response.content, "trace": ["[report] final report generated"]}

Patched stdio_client errlog for Colab.


In [6]:
from langgraph.graph import StateGraph, START, END

CONFIDENCE_THRESHOLD = 0.4
MAX_DIAGNOSTIC_ATTEMPTS = 2

def supervisor_router(state: AgentState) -> str:
    if "severity" in state and "root_causes" not in state and "action_plan" not in state:
        return "report" if state["severity"] == "low" else "diagnostic"

    if "root_causes" in state and "action_plan" not in state:
        confidence = state.get("diagnostic_confidence", 0.0)
        attempts = state.get("diagnostic_attempts", 0)
        if confidence < CONFIDENCE_THRESHOLD and attempts < MAX_DIAGNOSTIC_ATTEMPTS:
            return "diagnostic"
        return "planning" if state.get("root_causes") else "report"

    if "action_plan" in state:
        return "report"

    return "report"

graph = StateGraph(AgentState)
graph.add_node("triage", triage_agent)
graph.add_node("diagnostic", diagnostic_agent)
graph.add_node("planning", planning_agent)
graph.add_node("report", report_agent)

graph.add_edge(START, "triage")
graph.add_conditional_edges("triage", supervisor_router, {"report": "report", "diagnostic": "diagnostic"})
graph.add_conditional_edges("diagnostic", supervisor_router,
                             {"diagnostic": "diagnostic", "planning": "planning", "report": "report"})
graph.add_conditional_edges("planning", supervisor_router, {"report": "report"})
graph.add_edge("report", END)

app = graph.compile()
print("Graph compiled.")

Graph compiled.


In [7]:
async def run_alert(equipment_id: str, alert_text: str):
    initial_state = {
        "equipment_id": equipment_id,
        "alert_text": alert_text,
        "diagnostic_attempts": 0,
        "trace": [],
    }
    final_state = await app.ainvoke(initial_state)

    print("=" * 70)
    print(f"EQUIPMENT: {equipment_id}\nALERT: {alert_text}")
    print("-" * 70)
    print("TRACE:")
    for line in final_state["trace"]:
        print(" ", line)
    print("-" * 70)
    print("FINAL REPORT:\n")
    print(final_state["final_report"])
    print("=" * 70)
    return final_state

# High-severity path: full triage -> diagnostic(RAG) -> planning(MCP) -> report
await run_alert(
    "CB18-ST3",
    "Operator reports a loud grinding noise from the bearing housing on Station 3, "
    "along with a noticeable temperature increase at the housing over the last shift. "
    "Vibration sensor shows amplitude up roughly 3x baseline at shaft rotation frequency."
)

# Low-severity path: short-circuits straight to a log-only report
await run_alert(
    "CB28-ST1",
    "Minor cosmetic paint chip noticed on the pump housing guard. No performance impact reported."
)

/tmp/ipykernel_3619/2504317831.py:147: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(_llm(), tools)


EQUIPMENT: CB18-ST3
ALERT: Operator reports a loud grinding noise from the bearing housing on Station 3, along with a noticeable temperature increase at the housing over the last shift. Vibration sensor shows amplitude up roughly 3x baseline at shaft rotation frequency.
----------------------------------------------------------------------
TRACE:
  [triage] severity=high category=mechanical (The loud grinding noise and increased temperature at the bearing housing indicate a potential mechanical failure, which poses an imminent safety risk to the equipment's operation. The significant increase in vibration amplitude (3x baseline) further supports this assessment.)
  [diagnostic] attempt=1 top_cause=Loose or degraded mounting bolts allowing baseplate shift over time confidence=0.70
  [planning] action plan generated via MCP tools
  [report] final report generated
----------------------------------------------------------------------
FINAL REPORT:

Here's a concise, actionable report for 

{'equipment_id': 'CB28-ST1',
 'alert_text': 'Minor cosmetic paint chip noticed on the pump housing guard. No performance impact reported.',
 'severity': 'low',
 'fault_category': 'cosmetic',
 'diagnostic_attempts': 0,
 'final_report': 'Equipment: CB28-ST1\nAlert: Minor cosmetic paint chip noticed on the pump housing guard. No performance impact reported.\nSeverity: low — logged only, no action required.\nFault category: cosmetic',
 'trace': ['[triage] severity=low category=cosmetic (Minor cosmetic paint chip on the pump housing guard, no functional impact reported.)',
  '[report] low-severity, log-only report generated']}